# Flux LoRA Training — Colab T4

Тренировка кастомной LoRA конкретного человека с помощью **ai-toolkit** (Ostris) на бесплатном T4 GPU.

**Что вы получите:** LoRA файл (.safetensors), который можно использовать в ComfyUI с нодой LoraLoader для генерации изображений человека в любом стиле/окружении.

**Время:** ~40-70 мин (5-10 мин квантизация на CPU + 30-60 мин тренировка на GPU)
**VRAM:** ~6-12 GB на этапе тренировки. Квантизация модели происходит на CPU (это нормально — GPU подключается после).

---

### Как это работает (пошагово)

```
[1. GPU] → [2. HF Токен] → [3. Установка] → [4. ЗАГРУЗКА ФОТО] → [5. Тренировка] → [6. Тест] → [7. Скачать]
                                                     ↑
                                             Самый важный шаг!
                                        Загрузите 15-30 фото человека
                                        прямо сюда через кнопку upload
```

### Фазы тренировки (ячейка 5)

| Фаза | Время | CPU RAM | GPU VRAM | Что происходит |
|------|-------|---------|----------|----------------|
| **1. Квантизация** | ~5-10 мин | Высокий | Низкий | Модель сжимается с 24 ГБ до ~6 ГБ на CPU |
| **2. Кэширование** | ~1-3 мин | Средний | Средний | Изображения кодируются в латенты |
| **3. Тренировка** | ~30-50 мин | Низкий | **6-12 ГБ** | LoRA обучается на GPU |

> **Важно:** На фазе 1 GPU VRAM почти не используется — это нормальное поведение! GPU активно используется на фазе 3 (тренировка).

### Какие фото нужны для тренировки?

| Что нужно | Зачем |
|-----------|-------|
| **15-30 фото** одного человека | Чем больше, тем лучше модель запомнит лицо |
| **Разные ракурсы** (анфас, 3/4, профиль) | Модель учит лицо со всех сторон |
| **Разное освещение** (улица, помещение, студия) | Модель не привязывается к одному свету |
| **Разные выражения** (улыбка, серьёзное, смех) | Естественные результаты в генерации |
| **Чёткое лицо, хорошее качество** | Мусор на входе = мусор на выходе |

### Чего НЕ нужно

- Групповые фото (только один человек на фото!)
- Фото в солнечных очках / маске (лицо должно быть видно)
- Размытые или очень тёмные фото
- Фото только с одного ракурса

**Запускайте ячейки 1-7 по порядку.**

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
#@title 2. Авторизация Hugging Face (обязательно для FLUX)
#@markdown ---
#@markdown ### Зачем это нужно?
#@markdown Модель **FLUX.1-dev** от Black Forest Labs — **закрытая модель** на Hugging Face.
#@markdown Для скачивания нужно:
#@markdown 1. Зарегистрироваться на [huggingface.co](https://huggingface.co)
#@markdown 2. Принять лицензию на странице модели: [FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)
#@markdown 3. Создать токен: [Settings → Access Tokens](https://huggingface.co/settings/tokens) (тип: **Read**)
#@markdown 4. Вставить токен ниже

hf_token = "" #@param {type:"string"}
#@markdown Вставьте ваш Hugging Face токен (начинается с `hf_...`)

if not hf_token or not hf_token.strip().startswith("hf_"):
    print("=" * 60)
    print("  ОШИБКА: Токен не указан или неверный формат!")
    print()
    print("  1. Перейдите: https://huggingface.co/settings/tokens")
    print("  2. Создайте токен с доступом Read")
    print("  3. Вставьте его в поле hf_token выше")
    print("  4. Перезапустите эту ячейку")
    print("=" * 60)
    raise ValueError("Hugging Face токен обязателен для FLUX.1-dev")

!pip install huggingface_hub -q
from huggingface_hub import login
login(token=hf_token.strip())

import os
os.environ["HF_TOKEN"] = hf_token.strip()

print("\n" + "=" * 60)
print("  Авторизация успешна!")
print("  Теперь можно скачивать FLUX.1-dev")
print("=" * 60)

In [ ]:
#@title 3. Установка ai-toolkit + Зависимости
import psutil, shutil

# --- Swap для загрузки FLUX ---
# FLUX.1-dev ~24 ГБ в fp16, а у Colab T4 ~12-13 ГБ RAM.
# Без swap модель падает на "Loading checkpoint shards".
# Создаём 16 ГБ swap на диске — медленнее, но работает.

ram_gb = psutil.virtual_memory().total / 1024**3
swap_gb = psutil.swap_memory().total / 1024**3
disk_free_gb = shutil.disk_usage("/content").free / 1024**3

print(f"RAM: {ram_gb:.1f} ГБ | Swap: {swap_gb:.1f} ГБ | Диск свободно: {disk_free_gb:.1f} ГБ")

SWAP_SIZE_GB = 16
SWAP_PATH = "/content/swapfile"

if swap_gb < 8:
    if disk_free_gb < SWAP_SIZE_GB + 5:
        print(f"  ВНИМАНИЕ: Мало места на диске ({disk_free_gb:.0f} ГБ). Swap уменьшен до 8 ГБ.")
        SWAP_SIZE_GB = 8

    import os
    if not os.path.exists(SWAP_PATH):
        print(f"\nСоздание swap {SWAP_SIZE_GB} ГБ...")
        !sudo fallocate -l {SWAP_SIZE_GB}G {SWAP_PATH} && \
            sudo chmod 600 {SWAP_PATH} && \
            sudo mkswap {SWAP_PATH} && \
            sudo swapon {SWAP_PATH}

        new_swap = psutil.swap_memory().total / 1024**3
        print(f"Swap создан: {new_swap:.1f} ГБ")
        print(f"Итого виртуальной памяти: {ram_gb + new_swap:.1f} ГБ (нужно ~24 ГБ для FLUX)")
    else:
        # Swap файл уже есть, убедимся что активен
        !sudo swapon {SWAP_PATH} 2>/dev/null || true
        print(f"Swap уже существует и активен.")
else:
    print(f"Swap уже достаточный ({swap_gb:.1f} ГБ), создание не требуется.")

# --- Установка ai-toolkit ---
%cd /content
!git clone https://github.com/ostris/ai-toolkit.git 2>/dev/null || echo "Уже склонировано"
%cd /content/ai-toolkit
!git submodule update --init --recursive

# Обновляем pip/setuptools/wheel
!pip install --upgrade pip setuptools wheel -q

# Ставим зависимости
!pip install -r requirements.txt -q
!pip install peft accelerate safetensors PyYAML bitsandbytes -q
!pip install scipy jedi -q

# Показываем итоговые версии для диагностики
import numpy as np
print(f"\nnumpy версия: {np.__version__}")

# Итоговый отчёт о памяти
ram_gb = psutil.virtual_memory().total / 1024**3
swap_gb = psutil.swap_memory().total / 1024**3
print(f"\nПамять: RAM {ram_gb:.1f} ГБ + Swap {swap_gb:.1f} ГБ = {ram_gb + swap_gb:.1f} ГБ виртуальной")
if ram_gb + swap_gb < 24:
    print("  ВНИМАНИЕ: Менее 24 ГБ виртуальной памяти. Загрузка FLUX может быть нестабильной.")
else:
    print("  ОК: Достаточно для загрузки FLUX.1-dev (~24 ГБ)")
print("ai-toolkit установлен!")

In [ ]:
#@title 4. Загрузка тренировочных фото
#@markdown ## ⬇️ Нажмите кнопку "Выбрать файлы" ниже
#@markdown
#@markdown Загрузите **15-30 фотографий** человека, на которого будете тренировать LoRA.
#@markdown
#@markdown Фото сохранятся в папку `/content/dataset/` — именно оттуда
#@markdown тренировочный скрипт будет брать данные на следующем шаге.
#@markdown
#@markdown **Форматы:** .jpg, .jpeg, .png, .webp
#@markdown
#@markdown ---
#@markdown
#@markdown ### Чек-лист перед загрузкой:
#@markdown - ✅ На каждом фото **один человек**
#@markdown - ✅ Лицо **хорошо видно** (без очков, масок)
#@markdown - ✅ Есть фото с **разных ракурсов** (анфас, 3/4, профиль)
#@markdown - ✅ Есть фото при **разном освещении**
#@markdown - ✅ Фото **чёткие**, не размытые

from google.colab import files
from IPython.display import display, Image as IPImage
import os

DATASET_DIR = "/content/dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

print("=" * 60)
print("  ЗАГРУЗКА ФОТО ДЛЯ ТРЕНИРОВКИ LoRA")
print("=" * 60)
print(f"\n  Папка датасета: {DATASET_DIR}")
print("  Нажмите кнопку ниже и выберите 15-30 фото\n")

uploaded = files.upload()

VALID_EXT = ('.jpg', '.jpeg', '.png', '.webp')
saved = 0
for name, data in uploaded.items():
    if not name.lower().endswith(VALID_EXT):
        print(f"  Пропущено (неподдерживаемый формат): {name}")
        continue
    path = os.path.join(DATASET_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    saved += 1

# Показываем итог
all_files = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(VALID_EXT)]
count = len(all_files)

print(f"\n{'=' * 60}")
print(f"  Загружено: {saved} фото")
print(f"  Всего в датасете: {count} фото")
print(f"  Папка: {DATASET_DIR}")
print(f"{'=' * 60}")

if count < 10:
    print("\n  ⚠️  Менее 10 фото — результат может быть плохим.")
    print("  Рекомендуется: 15-30 фото для качественной LoRA.")
elif count <= 30:
    print(f"\n  ✅ Отлично! {count} фото — хороший датасет.")
else:
    print(f"\n  ℹ️  {count} фото — много, тренировка может занять дольше.")

# Превью загруженных фото
print(f"\nПревью (первые 8 фото):")
for f in sorted(all_files)[:8]:
    path = os.path.join(DATASET_DIR, f)
    print(f"  📷 {f}")
    try:
        display(IPImage(filename=path, width=200))
    except Exception:
        pass

In [ ]:
#@title 5. Настройка и запуск тренировки
#@markdown Задайте параметры тренировки ниже.

trigger_word = "ohwx" #@param {type:"string"}
#@markdown Триггер-слово — уникальный токен, который активирует LoRA. Используйте что-то редкое, например "ohwx" или "sks".

steps = 1500 #@param {type:"integer"}
#@markdown Количество шагов тренировки (рекомендуется 1000-2000 для T4)

learning_rate = 1e-4 #@param {type:"number"}
resolution = 512 #@param [512, 768] {type:"raw"}
#@markdown Разрешение тренировочных изображений. **512** — безопасный выбор для T4. **768** — лучше качество, но рискованнее по VRAM.
lora_rank = 16 #@param [4, 8, 16] {type:"raw"}
#@markdown LoRA rank — влияет на ёмкость модели. **16** — хороший баланс. При OOM уменьшите до **8**.
output_name = "my_person_lora" #@param {type:"string"}

import yaml, os, gc, torch, subprocess, threading, time, psutil

# --- Проверка HF токена ---
hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    print("=" * 60)
    print("  ОШИБКА: HF_TOKEN не найден!")
    print("  Запустите ячейку 2 (Авторизация HF) заново.")
    print("=" * 60)
    raise ValueError("HF_TOKEN не установлен — запустите ячейку 2")

# Сохраняем токен на диск, чтобы subprocess ai-toolkit его подхватил
from huggingface_hub import login
login(token=hf_token)
print("HF токен активен и сохранён на диск.")

# --- Проверка GPU ---
if not torch.cuda.is_available():
    print("CUDA не доступна! Тренировка на GPU невозможна.")
    raise RuntimeError("CUDA не найдена — проверьте, что Runtime → GPU включён")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
vram_total = torch.cuda.get_device_properties(0).total_memory
vram_gb = vram_total / 1024**3
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}, VRAM: {vram_gb:.1f} GiB")

if resolution == 768 and vram_gb < 14.0:
    print(f"\n  ВНИМАНИЕ: Разрешение 768 при {vram_gb:.1f} ГБ VRAM — рискованно.")
    print(f"  Если возникнет OOM, переключитесь на 512 и rank 8.\n")

# --- Проверка swap (обязательно для загрузки FLUX) ---
ram_gb_sys = psutil.virtual_memory().total / 1024**3
swap_gb_sys = psutil.swap_memory().total / 1024**3
total_virtual = ram_gb_sys + swap_gb_sys
print(f"RAM: {ram_gb_sys:.1f} ГБ + Swap: {swap_gb_sys:.1f} ГБ = {total_virtual:.1f} ГБ виртуальной")

if total_virtual < 24:
    print(f"\n  ВНИМАНИЕ: {total_virtual:.1f} ГБ виртуальной памяти — может не хватить для FLUX!")
    print(f"  FLUX.1-dev требует ~24 ГБ RAM для загрузки checkpoint shards.")
    print(f"  Создаём экстренный swap...")
    SWAP_PATH = "/content/swapfile"
    if not os.path.exists(SWAP_PATH):
        !sudo fallocate -l 16G {SWAP_PATH} && \
            sudo chmod 600 {SWAP_PATH} && \
            sudo mkswap {SWAP_PATH} && \
            sudo swapon {SWAP_PATH}
    else:
        !sudo swapon {SWAP_PATH} 2>/dev/null || true
    new_total = (psutil.virtual_memory().total + psutil.swap_memory().total) / 1024**3
    print(f"  Виртуальная память: {new_total:.1f} ГБ")

# --- Очистка системной RAM перед загрузкой модели ---
# Убиваем кеши и неиспользуемую память
gc.collect()
!sudo sh -c 'echo 3 > /proc/sys/vm/drop_caches' 2>/dev/null || true

ram_avail = psutil.virtual_memory().available / 1024**3
print(f"RAM доступно: {ram_avail:.1f} ГБ (после очистки кешей)")

# Борьба с фрагментацией памяти CUDA
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Конфиг тренировки ---
# Архитектура памяти для T4 (16 ГБ VRAM):
#
# ПРОБЛЕМА: FLUX.1-dev = ~24 ГБ fp16. T4 = 16 ГБ VRAM, ~12 ГБ RAM.
#
# РЕШЕНИЕ (цепочка оптимизаций):
# 1. Swap 16 ГБ на диске → модель влезает в виртуальную память при загрузке
# 2. low_vram: True → квантизация на CPU (не GPU), компоненты на CPU
# 3. quantize: True → qfloat8, ~24 ГБ → ~12 ГБ
# 4. layer_offloading: 80% → стриминг слоёв CPU↔GPU через ping-pong буферы
# 5. gradient_checkpointing → экономия VRAM на активациях
# 6. dtype: fp16 → T4 (Turing) НЕ поддерживает bf16!

config = {
    "job": "extension",
    "config": {
        "name": output_name,
        "process": [{
            "type": "sd_trainer",
            "training_folder": "/content/output",
            "performance_log_every": 100,
            "device": "cuda:0",
            "trigger_word": trigger_word,
            "low_vram": True,
            "network": {
                "type": "lora",
                "linear": lora_rank,
                "linear_alpha": lora_rank
            },
            "save": {
                "dtype": "float16",
                "save_every": 500,
                "max_step_saves_to_keep": 2
            },
            "datasets": [{
                "folder_path": "/content/dataset",
                "caption_ext": "txt",
                "caption_dropout_rate": 0.05,
                "shuffle_tokens": False,
                "cache_latents_to_disk": True,
                "resolution": [resolution, resolution]
            }],
            "train": {
                "batch_size": 1,
                "steps": steps,
                "gradient_accumulation_steps": 1,
                "train_unet": True,
                "train_text_encoder": False,
                "gradient_checkpointing": True,
                "noise_scheduler": "flowmatch",
                "optimizer": "adamw8bit",
                "lr": learning_rate,
                "ema_config": {"use_ema": False},
                "dtype": "fp16"
            },
            "model": {
                "name_or_path": "black-forest-labs/FLUX.1-dev",
                "is_flux": True,
                "quantize": True,
                "layer_offloading": True,
                "layer_offloading_transformer_percent": 0.8
            },
            "sample": {
                "sampler": "flowmatch",
                "sample_every": 500,
                "width": 512,
                "height": 512,
                "prompts": [
                    f"a professional photo of {trigger_word} person, studio lighting, 4k"
                ],
                "neg": "",
                "seed": 42,
                "walk_seed": True,
                "guidance_scale": 3.5,
                "sample_steps": 20
            }
        }]
    }
}

config_path = "/content/ai-toolkit/config/train_lora.yaml"
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"\nКонфиг сохранён: {config_path}")
print(f"\nПараметры тренировки:")
print(f"  Триггер-слово: {trigger_word}")
print(f"  Шаги: {steps}")
print(f"  Скорость обучения: {learning_rate}")
print(f"  Разрешение: {resolution}x{resolution}")
print(f"  LoRA rank: {lora_rank}")
print(f"  dtype: fp16 (T4 не поддерживает bf16)")
print(f"  Квантизация: qfloat8 (~12 ГБ вместо ~24 ГБ)")
print(f"  low_vram: True (квантизация на CPU)")
print(f"  layer_offloading: 80% (стриминг слоёв CPU<->GPU)")
print(f"  Gradient checkpointing: включён")
print(f"  Сэмплирование: каждые 500 шагов (512x512)")

print(f"\n{'='*60}")
print(f"  ФАЗЫ ТРЕНИРОВКИ:")
print(f"  1. Загрузка checkpoint shards (~3-7 мин)")
print(f"     -> Использует RAM + swap. GPU VRAM не задействован.")
print(f"     -> Если swap недостаточен — будет SIGKILL (перезапустите ячейку 3).")
print(f"  2. Квантизация на CPU (~3-5 мин)")
print(f"     -> Модель сжимается с ~24 ГБ до ~12 ГБ (qfloat8)")
print(f"  3. Кэширование латентов (~1-3 мин)")
print(f"  4. Тренировка LoRA на GPU (~30-50 мин)")
print(f"     -> GPU VRAM вырастет, слои стримятся с CPU")
print(f"{'='*60}")

# --- Фоновый мониторинг GPU VRAM ---
vram_log_path = "/content/vram_monitor.log"
stop_monitor = threading.Event()

def monitor_vram():
    """Логирует VRAM каждые 30 сек в файл для диагностики."""
    with open(vram_log_path, "w") as log:
        while not stop_monitor.is_set():
            try:
                result = subprocess.run(
                    ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
                     "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, timeout=5
                )
                if result.returncode == 0:
                    parts = result.stdout.strip().split(", ")
                    if len(parts) == 3:
                        used_mb, total_mb, gpu_util = parts
                        ts = time.strftime("%H:%M:%S")
                        line = f"[{ts}] VRAM: {used_mb}/{total_mb} MiB ({gpu_util}% GPU util)"
                        log.write(line + "\n")
                        log.flush()
            except Exception:
                pass
            stop_monitor.wait(30)

monitor_thread = threading.Thread(target=monitor_vram, daemon=True)
monitor_thread.start()
print(f"\nМониторинг GPU запущен (лог: {vram_log_path})")

print(f"\nЗапуск тренировки...\n")

# --- Запуск тренировки ---
!cd /content/ai-toolkit && \
    CUDA_VISIBLE_DEVICES=0 \
    HF_TOKEN="{hf_token}" \
    HUGGING_FACE_HUB_TOKEN="{hf_token}" \
    PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python run.py config/train_lora.yaml

# --- Останавливаем мониторинг ---
stop_monitor.set()
monitor_thread.join(timeout=5)

# --- Итоговый отчёт по GPU ---
print(f"\n{'='*60}")
print(f"  ИТОГИ ТРЕНИРОВКИ — ИСПОЛЬЗОВАНИЕ GPU")
print(f"{'='*60}")

# Показываем лог VRAM
if os.path.exists(vram_log_path):
    with open(vram_log_path) as f:
        lines = f.readlines()
    if lines:
        print(f"\n  Записей в логе VRAM: {len(lines)}")
        max_vram = 0
        max_line = ""
        for line in lines:
            try:
                used = int(line.split("VRAM: ")[1].split("/")[0])
                if used > max_vram:
                    max_vram = used
                    max_line = line.strip()
            except (IndexError, ValueError):
                pass
        if max_vram > 0:
            print(f"  Пиковое VRAM: {max_vram} MiB ({max_vram/1024:.1f} GiB)")
            print(f"  {max_line}")

        print(f"\n  Начало:")
        for line in lines[:3]:
            print(f"    {line.strip()}")
        if len(lines) > 6:
            print(f"  ...")
        print(f"  Конец:")
        for line in lines[-3:]:
            print(f"    {line.strip()}")
    else:
        print("  Лог VRAM пуст — nvidia-smi не вернула данных")

# Текущее состояние GPU
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

print(f"{'='*60}")
print(f"\nТренировка завершена!")

In [ ]:
#@title 6. Тест LoRA — Генерация тестового фото
#@markdown Быстрый тест обученной LoRA через пайплайн diffusers.

trigger_word = "ohwx" #@param {type:"string"}
prompt = "a photo of ohwx person, professional headshot, studio lighting, 4k" #@param {type:"string"}
output_name = "my_person_lora" #@param {type:"string"}

import glob, os

# Находим последний LoRA файл
lora_files = sorted(glob.glob(f"/content/output/{output_name}/*.safetensors"))
if not lora_files:
    print("LoRA файл не найден! Убедитесь, что тренировка завершена.")
else:
    lora_path = lora_files[-1]
    print(f"Используется LoRA: {lora_path}")

    import torch
    from diffusers import FluxPipeline

    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        print("HF_TOKEN не найден! Запустите ячейку 2 (Авторизация HF) заново.")
        raise ValueError("HF_TOKEN не установлен")

    pipe = FluxPipeline.from_pretrained(
        "black-forest-labs/FLUX.1-dev",
        torch_dtype=torch.float16,
        token=hf_token
    )
    pipe.enable_model_cpu_offload()
    pipe.load_lora_weights(lora_path)

    image = pipe(
        prompt,
        num_inference_steps=25,
        guidance_scale=3.5,
        generator=torch.Generator("cuda").manual_seed(42)
    ).images[0]

    image.save("/content/test_result.png")

    from IPython.display import display, Image
    display(Image("/content/test_result.png"))
    print("Тестовое изображение сохранено: /content/test_result.png")

    del pipe
    import gc; gc.collect()
    torch.cuda.empty_cache()

In [ ]:
#@title 7. Скачивание LoRA / Сохранение на Google Drive
#@markdown Выберите, куда сохранить обученный LoRA файл.

save_to_drive = True #@param {type:"boolean"}
output_name = "my_person_lora" #@param {type:"string"}

import glob, shutil, os

lora_files = sorted(glob.glob(f"/content/output/{output_name}/*.safetensors"))
if not lora_files:
    print("LoRA файл не найден!")
else:
    lora_path = lora_files[-1]
    lora_filename = os.path.basename(lora_path)
    print(f"LoRA файл: {lora_path} ({os.path.getsize(lora_path) / 1024**2:.1f} MB)")

    if save_to_drive:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_dir = "/content/drive/MyDrive/ComfyUI_LoRAs"
        os.makedirs(drive_dir, exist_ok=True)
        dest = f"{drive_dir}/{lora_filename}"
        shutil.copy2(lora_path, dest)
        print(f"Сохранено на Drive: {dest}")
    else:
        from google.colab import files
        files.download(lora_path)
        print("Скачивание начато!")

<cell_type>markdown</cell_type>---
## Использование LoRA в ComfyUI

### Способ 1: Нода LoraLoader
1. Скопируйте LoRA файл в `ComfyUI/models/loras/`
2. В вашем воркфлоу добавьте ноду **LoraLoader**
3. Подключите её между загрузчиком модели и KSampler
4. Выберите ваш LoRA файл
5. Используйте триггер-слово в промпте (например, "a photo of ohwx person in a garden")

### Способ 2: В Colab
Используйте ячейку "Скачать LoRA" в `colab_flux_photo.ipynb` для прямой загрузки.

### Советы
- **Триггер-слово** должно присутствовать в промпте для активации LoRA
- **Сила (strength)** 0.7-1.0 лучше всего работает для LoRA лиц
- **Пониженная сила** (0.4-0.6) для большего стилистического разнообразия
- Комбинируйте с IPAdapter FaceID для ещё лучшей консистентности

### Решение проблем

| Проблема | Причина | Решение |
|----------|---------|---------|
| **Падает на "Loading checkpoint shards"** | Недостаточно системной RAM (~12 ГБ на Colab) для загрузки FLUX (~24 ГБ) | Перезапустите ячейку 3 — она создаёт 16 ГБ swap на диске |
| **SIGKILL без ошибки** | Ядро убивает процесс из-за нехватки RAM | То же — нужен swap. Если не помогает, перезагрузите Runtime |
| **GPU VRAM не используется в начале** | Фазы 1-2 работают на CPU (квантизация) | Это нормально! GPU подключается на фазе тренировки |
| **OOM (Out of Memory) на GPU** | VRAM не хватает для тренировки | Уменьшите resolution до 512 и LoRA rank до 8 |
| **dtype mismatch / bf16 ошибка** | T4 не поддерживает bf16 | Убедитесь что dtype: fp16 (уже установлено) |
| **Переобучение (все изображения одинаковые)** | Слишком много шагов / мало данных | Уменьшите steps, добавьте больше фото |
| **Недообучение (лицо не распознаётся)** | Мало шагов / плохие фото | Увеличьте steps, улучшите качество датасета |
| **Медленная тренировка** | layer_offloading стримит слои через PCIe | Нормально для T4 — ~6-10 сек/шаг. Можно уменьшить steps |